In [1]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB


In [3]:
df.isna().sum()

,0
transaction_id,0
date,0
region,3
product_category,2
sales_amount,4
quantity,3
customer_age,4
payment_method,3


In [4]:
df.describe()

,transaction_id,date,sales_amount,quantity,customer_age
count,20.00000,20,16.000000,17.000000,16.000000
mean,10.50000,2024-10-10 12:00:00,891.250000,2.705882,36.812500
min,1.00000,2024-10-01 00:00:00,340.000000,1.000000,25.000000
25%,5.75000,2024-10-05 18:00:00,495.000000,2.000000,30.500000
50%,10.50000,2024-10-10 12:00:00,725.000000,3.000000,36.500000
75%,15.25000,2024-10-15 06:00:00,1125.000000,4.000000,42.500000
max,20.00000,2024-10-20 00:00:00,2100.000000,5.000000,52.000000
std,5.91608,NaN,526.546927,1.358524,8.158584


In [13]:
df["region"] = df["region"].fillna(df["region"].mode()[0])
df["product_category"] = df["product_category"].fillna(df["product_category"].mode()[0])
df["sales_amount"] = df["sales_amount"].fillna(df["sales_amount"].mean())
df["quantity"] = df["quantity"].fillna(method='ffill')
df["customer_age"] = df["customer_age"].fillna(df["customer_age"].mean().round(2))
df1 = df.dropna(subset="payment_method")
df1.isna().sum()

/tmp/ipykernel_232/60545937.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["quantity"] = df["quantity"].fillna(method='ffill')


,0
transaction_id,0
date,0
region,0
product_category,0
sales_amount,0
quantity,0
customer_age,0
payment_method,0


In [14]:
total_sales = df1.groupby("region")["sales_amount"].sum()
print(total_sales)

region
East     3790.0
North    6460.0
South    2232.5
West     2562.5
Name: sales_amount, dtype: float64


In [16]:
average_sales = df1.groupby("product_category")["sales_amount"].mean().round(2)
print(average_sales)

product_category
Books           563.75
Clothing        735.42
Electronics    1422.81
Home            812.50
Name: sales_amount, dtype: float64


In [23]:
# total sales and quantity
result = df1.groupby(["region", "product_category"]).agg({"sales_amount": "sum","quantity":"sum"}).reset_index()
print(result)

  region product_category  sales_amount  quantity
0   East            Books        800.00       4.0
1   East         Clothing        890.00       3.0
2   East      Electronics       2100.00       1.0
3  North         Clothing        510.00       3.0
4  North      Electronics       2700.00       3.0
5  North             Home       3250.00      12.0
6  South         Clothing       2232.50       9.0
7   West            Books        891.25       1.0
8   West         Clothing        780.00       5.0
9   West      Electronics        891.25       1.0


In [33]:
top_3 = df1.groupby(["region","product_category"]).agg(total_sales = ("sales_amount","sum")).sort_values("total_sales", ascending = False).reset_index()
top_3.head(3)

,region,product_category,total_sales
0,North,Home,3250.0
1,North,Electronics,2700.0
2,South,Clothing,2232.5


In [30]:
def sales_range(x):
  return x.max()-x.min()

aggregate = df1.groupby("region")["sales_amount"].agg(sales_range)
print(aggregate)

region
East     1720.00
North     990.00
South     441.25
West      111.25
Name: sales_amount, dtype: float64


In [32]:
aggregate1 = df1.groupby("region").agg({"sales_amount": ["sum", "mean", "max"], "quantity": ["sum", "min"]})
print(aggregate1)

       sales_amount                      quantity     
                sum        mean      max      sum  min
region                                                
East         3790.0  947.500000  2100.00      8.0  1.0
North        6460.0  922.857143  1500.00     18.0  1.0
South        2232.5  744.166667   891.25      9.0  2.0
West         2562.5  854.166667   891.25      7.0  1.0
